### **Storage and Reading Files with Pandas 1**

Reading the CSV file directly from the zip archive using `pandas.read_csv` is typically faster than first unpacking the file and then reading it.

This is because unpacking requires an additional step where the data is written to disk before being read again, resulting in extra disk I/O. Disk operations are relatively slow and therefore increase the total execution time.

In contrast, when reading directly from the zip file, Pandas decompresses the data in memory and processes it in a single step, avoiding unnecessary intermediate writes to disk.

However, if the same dataset needs to be read multiple times, unpacking it once may be beneficial, as it avoids repeated decompression overhead.

### **Storage and Reading Files with Pandas 2**

In [55]:
import pandas as pd

df = pd.read_csv('data/2023_01.csv')

In [ ]:
def df_memsize(df):
    return df.memory_usage(deep=True).sum()

df_memsize(df)

np.int64(1949018494)

### **Storage and Reading Files with Pandas 3**

In [24]:
def summarize_columns(df):
    print(pd.DataFrame([
        (
            c,
            df[c].dtype,
            len(df[c].unique()),
            df[c].memory_usage(deep=True) // (1024**2)
        ) for c in df.columns
    ], columns=['name', 'dtype', 'unique', 'size (MB)']))
    print('Total size:', df.memory_usage(deep=True).sum() / 1024**2, 'MB')

summarize_columns(df)

          name    dtype   unique  size (MB)
0      coordsx  float64      224         62
1      coordsy  float64      219         62
2      created      str  8142495        590
3     observed      str    44640        535
4  parameterId      str       47        484
5    stationId    int64      247         62
6        value  float64    11532         62
Total size: 1858.7288799285889 MB


Some things we can do to reduce memory is to convert object columns to categorical, or the objects to datetime, and objects here are the three columns with "str". Furthermore, we can reduce floats from $64$ to $32$ for instance. We can also downcast the interger.

### **Storage and Reading Files with Pandas 4**

In [25]:
def reduce_dmi_df(df):
    df['parameterId'] = df['parameterId'].astype('category')
    df["created"] = pd.to_datetime(df["created"], format="ISO8601")
    df["observed"] = pd.to_datetime(df["observed"], format="ISO8601")
    df["coordsx"] = df["coordsx"].astype('float32')
    df["coordsy"] = df["coordsy"].astype('float32')
    df["value"] = df["value"].astype('float32')
    df["stationId"] = df["stationId"].astype('int32')
    return df

In [26]:
reduce_dmi_df(df)

,coordsx,coordsy,created,observed,parameterId,stationId,value
0,9.787500,56.149601,2023-07-07 21:57:06.803045+00:00,2023-01-17 23:59:00+00:00,precip_past1min,5185,0.0
1,8.146500,56.503201,2023-07-07 21:57:44.125851+00:00,2023-01-17 23:59:00+00:00,precip_past1min,5296,0.0
2,9.287700,57.014301,2023-07-07 21:56:34.840129+00:00,2023-01-17 23:59:00+00:00,precip_past1min,5085,0.0
3,8.665200,54.987202,2023-07-07 21:58:08.833412+00:00,2023-01-17 23:59:00+00:00,precip_past1min,5355,0.0
4,12.098100,55.207600,2023-07-07 21:59:11.626531+00:00,2023-01-17 23:59:00+00:00,precip_past1min,5889,0.0
...,...,...,...,...,...,...,...
8142490,14.749400,55.067699,2023-07-08 07:41:47.665513+00:00,2023-01-01 00:00:00+00:00,wind_dir_past1h,6190,209.0
8142491,-51.730801,64.183296,2023-07-08 07:41:47.637314+00:00,2023-01-01 00:00:00+00:00,temp_min_past1h,4250,-9.2
8142492,10.127200,56.302700,2023-07-07 17:20:39.101115+00:00,2023-01-01 00:00:00+00:00,precip_past10min,6072,0.5
8142493,9.404100,56.879002,2023-07-07 20:15:09.826122+00:00,2023-01-01 00:00:00+00:00,precip_dur_past10min,5081,10.0


In [27]:
summarize_columns(df)

          name                dtype   unique  size (MB)
0      coordsx              float32      224         31
1      coordsy              float32      219         31
2      created  datetime64[us, UTC]  8142495         62
3     observed  datetime64[us, UTC]    44640         62
4  parameterId             category       47          7
5    stationId                int32      247         31
6        value              float32    11532         31
Total size: 256.25750160217285 MB


### **Reading Files with Arrow 1**

In [56]:
def pyarrow_load(fname):
    from pyarrow import csv
    table = csv.read_csv(fname)
    return table

In [57]:
pyarrow_df = pyarrow_load("data/2023_01.csv")

### **Reading Files with Arrow 2**

In [58]:
def pyarrow_load(fname):
    from pyarrow import csv
    table = csv.read_csv(fname)
    return table.to_pandas()

In [59]:
df = pyarrow_load("data/2023_01.csv")

### **Reading Files with Arrow 3**

In [61]:
# First we check for pyarrow
print(pyarrow_df.nbytes / 1024**2, "MB")

507.56864738464355 MB


In [62]:
# Then for pyarrow TO pandas df
summarize_columns(df)

          name                dtype   unique  size (MB)
0      coordsx              float64      224         62
1      coordsy              float64      219         62
2      created  datetime64[ns, UTC]  8142495         62
3     observed   datetime64[s, UTC]    44640         62
4  parameterId                  str       47        484
5    stationId                int64      247         62
6        value              float64    11532         62
Total size: 857.0067491531372 MB


We see that the pyarrow to pandas DF is 350 MB more than just pyarrow table.

The PyArrow table is smaller because it uses a compact columnar memory format and efficient encoding for strings. When converting to Pandas, string columns are stored as Python objects, which introduces significant overhead and increases memory usage.

### **Reading Files with Arrow 4**

In [63]:
def pyarrow_load(fname):
    from pyarrow import csv
    return csv.read_csv(fname).to_pandas()

def reduce_dmi_df(df):
    df['parameterId'] = df['parameterId'].astype('category')
    df["created"] = pd.to_datetime(df["created"], format="ISO8601")
    df["observed"] = pd.to_datetime(df["observed"], format="ISO8601")
    df["coordsx"] = df["coordsx"].astype('float32')
    df["coordsy"] = df["coordsy"].astype('float32')
    df["value"] = df["value"].astype('float32')
    df["stationId"] = df["stationId"].astype('int32')
    return df

def load_and_reduce(fname):
    df = pyarrow_load(fname)
    return reduce_dmi_df(df)

In [67]:
pyrarrow_df_reduced = load_and_reduce("data/2023_01.csv")

summarize_columns(pyrarrow_df_reduced)

          name                dtype   unique  size (MB)
0      coordsx              float32      224         31
1      coordsy              float32      219         31
2      created  datetime64[ns, UTC]  8142495         62
3     observed   datetime64[s, UTC]    44640         62
4  parameterId             category       47          7
5    stationId                int32      247         31
6        value              float32    11532         31
Total size: 256.25750160217285 MB


Went down from 857 MB to 256 MB, but this is very similar to just the pandas df from before.

### **Parquet Files 1**

Look in `parquet_files.py`.

### **Parquet Files 2**

In [68]:
def parquet_files(filename):
    from pyarrow import csv
    import pyarrow.parquet as pq

    table = csv.read_csv(filename)
    pq.write_table(table, filename.replace(".csv", ".parquet"))

    return table

filename = "data/2023_01.csv"

parquet_files(filename)

pyarrow.Table
coordsx: double
coordsy: double
created: timestamp[ns, tz=UTC]
observed: timestamp[s, tz=UTC]
parameterId: string
stationId: int64
value: double
----
coordsx: [[9.7875,8.1465,9.2877,8.6652,12.0981,...,11.1348,11.9435,10.8694,9.5067,12.7114],[11.3285,9.5067,11.3292,10.1009,9.7875,...,10.5466,11.8648,10.4249,10.1644,8.4871],...,[11.3285,10.1272,10.4398,10.5466,9.9527,...,11.8648,8.6242,8.3207,-45.44,8.5599],[10.7094,-21.9511,12.5263,-46.0294,12.4218,...,14.7494,-51.7308,10.1272,9.4041,11.6035]]
coordsy: [[56.1496,56.5032,57.0143,54.9872,55.2076,...,55.1593,54.5687,55.7435,56.7558,55.5364],[55.2465,56.7558,54.8275,55.0978,56.1496,...,56.1103,55.4026,55.0721,56.7157,56.1837],...,[55.2465,56.3027,55.3088,56.1103,57.1852,...,55.4026,55.9591,56.7637,61.1575,55.1904],[54.7567,70.4844,55.7664,60.7131,55.2931,...,55.0677,64.1833,56.3027,56.879,55.7358]]
created: [[2023-07-07 21:57:06.803045000Z,2023-07-07 21:57:44.125851000Z,2023-07-07 21:56:34.840129000Z,2023-07-07 21:58:08.833412

It is faster than reading the pandas file directly?

In [69]:
# OR WE CAN DO THIS?

import time
from pyarrow import csv
import pyarrow.parquet as pq

filename = "data/2023_01.csv"

# Measure Parquet write (includes reading CSV)
start = time.time()
table = csv.read_csv(filename)
pq.write_table(table, "data/test.parquet")
parquet_write_time = time.time() - start

# Measure Parquet read
start = time.time()
table = pq.read_table("data/test.parquet")
parquet_read_time = time.time() - start

print("Parquet write time:", parquet_write_time)
print("Parquet read time:", parquet_read_time)

Parquet write time: 1.6876537799835205
Parquet read time: 3.214444160461426


In [70]:
# WHEN DIRECTLY READING THE CSV WITH PANDAS

filename = "data/2023_01.csv"

# Measure CSV read
start = time.time()
df = pd.read_csv(filename)
csv_read_time = time.time() - start

# Measure CSV write
start = time.time()
df.to_csv("data/test.csv", index=False)
csv_write_time = time.time() - start

print("CSV read time:", csv_read_time)
print("CSV write time:", csv_write_time)

CSV read time: 6.375895023345947
CSV write time: 18.66159224510193


### **Pandas - Fast Operations 1**

In [71]:
def total_precip(df):
    total = 0.0
    for i in range(len(df)):
        row = df.iloc[i]
        if row['parameterId'] == 'precip_past10min':
            total += row['value']
    return total

In [82]:
# we use pandas.sample to get a smaller dataframe for testing
df_sample = df.sample(10000, random_state=42)

start = time.time()
res_loop = total_precip(df_sample)
loop_time = time.time() - start

print(loop_time)

0.1639690399169922


### **Pandas - Fast Operations 2**

In [83]:
# now we use apply() method
def total_precip_apply(df):
    total = df.apply(
        lambda row: row['value'] if row['parameterId'] == 'precip_past10min' else 0,
        axis=1
    ).sum()
    return total

start = time.time()
res_apply = total_precip_apply(df_sample)
apply_time = time.time() - start

print(apply_time)

0.022520065307617188


There was no speedup...

### **Pandas - Fast Operations 3**

In [84]:
def total_precip_vectorized(df):
    return df.loc[df['parameterId'] == 'precip_past10min', 'value'].sum()

start = time.time()
res_vec = total_precip_vectorized(df_sample)
vec_time = time.time() - start

print(vec_time)

0.0014863014221191406


### **Pandas - Fast Operations 4**

See the `fast_ops.py` file for the code.

### **Pandas - Fast Operations 5**

Idk.